# Pilot Data Collection — Bluesky

**Project:** *The Devil Wears Prada 2* — Social Media Analysis
**Platform:** Bluesky (AT Protocol via `atproto` SDK)

This notebook collects a pilot sample of Bluesky posts about *The Devil Wears Prada 2*. The pilot covers twelve search queries targeting the film, its cast, the characters, and the press-tour guests, restricted to a three-month window around the release date (1 May 2026). Its purpose is to verify that the topic produces enough posts, authors, and reply activity to support the downstream network and content analyses.

For each post we extract the standard text and engagement metadata, the language tags, the facets (hashtags, mentions, links), and the reply-threading references (`reply_root_uri`, `reply_parent_uri`) needed to reconstruct conversation graphs.


## 1. Setup

In [1]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from pandas.io.parquet import to_parquet
from atproto import Client, models
from atproto_client.exceptions import (
    RequestException,
    InvokeTimeoutError,
    NetworkError,
)

Credentials are loaded from a local `credentials.json` file, which is excluded from version control.

In [2]:
with open("../credentials.json", "r") as f:
    credentials = json.load(f)

client = Client()
client.login(credentials["handle"], credentials["app_password"])
print("Logged in as:", credentials["handle"])


Logged in as: martapaniconi.bsky.social


## 2. Helper functions

`_parse_dt_utc` parses Bluesky's ISO 8601 timestamps into UTC datetimes.

`call_with_retries` wraps every API call. It retries on transient network errors and timeouts with exponential backoff, and on HTTP 429 it honours the `ratelimit-reset` and `retry-after` response headers so we wait exactly the time the server requires.

`extract_facets` reads the rich-text facets attached to each post and returns the embedded hashtags, mentioned user DIDs, and external links.


In [3]:
def _parse_dt_utc(iso_str: str) -> datetime:
    dt = datetime.fromisoformat(iso_str.replace("Z", "+00:00"))
    if dt.tzinfo is None:
        return dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def _now_utc_ts() -> int:
    return int(datetime.now(timezone.utc).timestamp())


def call_with_retries(callable_fn, *args, max_retries: int = 10, **kwargs):
    for attempt in range(max_retries):
        try:
            return callable_fn(*args, **kwargs)

        except InvokeTimeoutError:
            wait_s = min(60, 2 ** attempt)
            print(f"[timeout] sleeping {wait_s}s")
            time.sleep(wait_s)
            continue

        except NetworkError:
            wait_s = min(60, 2 ** attempt)
            print(f"[network] sleeping {wait_s}s")
            time.sleep(wait_s)
            continue

        except RequestException as e:
            resp = getattr(e, "response", None)
            status = getattr(resp, "status_code", None)
            headers = getattr(resp, "headers", {}) or {}

            if status == 429:
                reset = headers.get("ratelimit-reset") or headers.get("RateLimit-Reset")
                retry_after = headers.get("retry-after") or headers.get("Retry-After")
                if reset:
                    wait_s = max(1, int(reset) - _now_utc_ts()) + 1
                elif retry_after:
                    wait_s = int(float(retry_after))
                else:
                    wait_s = min(60, 2 ** attempt)
                print(f"[429] rate limited, sleeping {wait_s}s")
                time.sleep(wait_s)
                continue

            wait_s = min(20, 2 ** attempt)
            print(f"[{status}] request failed, sleeping {wait_s}s")
            time.sleep(wait_s)

    raise RuntimeError("Too many retries / repeated failures.")


In [4]:
def extract_facets(record) -> dict:
    """Extract hashtags, mentions (DIDs), and external links from a post record."""
    hashtags, mentions, links = [], [], []

    for facet in (getattr(record, "facets", None) or []):
        for feat in (getattr(facet, "features", None) or []):
            ftype = (
                getattr(feat, "py_type", None)
                or getattr(feat, "type", None)
                or getattr(feat, "$type", None)
            )
            if ftype is None and isinstance(feat, dict):
                ftype = feat.get("$type") or feat.get("type")

            get = (lambda k: feat.get(k)) if isinstance(feat, dict) else (lambda k: getattr(feat, k, None))

            if ftype == "app.bsky.richtext.facet#tag" and get("tag"):
                hashtags.append(get("tag"))
            elif ftype == "app.bsky.richtext.facet#mention" and get("did"):
                mentions.append(get("did"))
            elif ftype == "app.bsky.richtext.facet#link" and get("uri"):
                links.append(get("uri"))

    return {"hashtags": hashtags, "mentions": mentions, "links": links}


## 3. Collection function

`collect_search_posts` performs the search for a single query. It uses Bluesky's `searchPosts` endpoint with cursor-based pagination, sorts results by latest, and applies the time window both server-side (via `since`/`until`) and as a safety check on each result.

For every returned post we flatten the fields we need: identifiers, timestamps, text, author, engagement counters, language tags, facets, reply-threading references, embed type, and moderation labels. Each row is tagged with the originating query so we can later analyse per-query yield.


In [5]:
def collect_search_posts(
    client,
    q: str,
    limit_total: int = 100,
    page_size: int = 25,
    since_iso: str | None = None,
    until_iso: str | None = None,
    polite_sleep: float = 0.6,
):
    """Collect posts matching `q` with pagination, facets, and reply threading."""
    if not isinstance(q, str) or not q.strip():
        raise ValueError("q must be a non-empty string")

    cursor, rows = None, []
    since_dt = _parse_dt_utc(since_iso) if since_iso else None
    until_dt = _parse_dt_utc(until_iso) if until_iso else None

    while len(rows) < limit_total:
        params = models.AppBskyFeedSearchPosts.Params(
            q=q,
            sort="latest",
            since=since_iso,
            until=until_iso,
            limit=min(page_size, limit_total - len(rows)),
            cursor=cursor,
        )
        res = call_with_retries(client.app.bsky.feed.search_posts, params)
        cursor = res.cursor
        posts = res.posts or []

        if not posts:
            break

        for p in posts:
            rec = getattr(p, "record", None)
            if rec is None:
                continue

            created = getattr(rec, "created_at", None)
            created_dt = _parse_dt_utc(created) if created else None
            if created_dt is not None:
                if since_dt and created_dt < since_dt:
                    continue
                if until_dt and created_dt >= until_dt:
                    continue

            author = getattr(p, "author", None)
            facets = extract_facets(rec)

            reply_obj = getattr(rec, "reply", None)
            reply_root = reply_parent = None
            if reply_obj is not None:
                root_ref = getattr(reply_obj, "root", None)
                parent_ref = getattr(reply_obj, "parent", None)
                reply_root = getattr(root_ref, "uri", None) if root_ref else None
                reply_parent = getattr(parent_ref, "uri", None) if parent_ref else None

            embed = getattr(rec, "embed", None) or getattr(p, "embed", None)
            embed_type = (
                getattr(embed, "py_type", None)
                or getattr(embed, "$type", None)
                or getattr(embed, "type", None)
            )

            labels = []
            for lab in (getattr(p, "labels", None) or getattr(rec, "labels", None) or []):
                val = getattr(lab, "val", None) or (lab.get("val") if isinstance(lab, dict) else None)
                if val:
                    labels.append(val)

            rows.append({
                "query": q,
                "uri": getattr(p, "uri", None),
                "cid": getattr(p, "cid", None),
                "created_at": created_dt,
                "text": getattr(rec, "text", None),
                "author_handle": getattr(author, "handle", None),
                "author_did": getattr(author, "did", None),
                "author_display_name": getattr(author, "display_name", None),
                "reply_count": getattr(p, "reply_count", None),
                "repost_count": getattr(p, "repost_count", None),
                "like_count": getattr(p, "like_count", None),
                "quote_count": getattr(p, "quote_count", None),
                "hashtags": facets["hashtags"],
                "mentions": facets["mentions"],
                "links": facets["links"],
                "langs": getattr(rec, "langs", None),
                "reply_root_uri": reply_root,
                "reply_parent_uri": reply_parent,
                "embed_type": embed_type,
                "labels": labels,
            })

            if len(rows) >= limit_total:
                break

        if cursor is None:
            break
        time.sleep(polite_sleep)

    return rows


## 4. Queries and time window

Twelve queries cover four overlapping audiences: the film itself, the two main characters (Miranda Priestly and her real-world referent Anna Wintour), the four returning cast members, and two guest figures associated with the press tour (Lady Gaga and Doechii). Cast names are disambiguated by pairing them with `prada` so that we capture posts about the film rather than every mention of the actor.

The window starts on 1 March 2026, two months before release, to include the marketing build-up, and ends on 21 May 2026, the date of this collection.


In [6]:
queries = [
    '"the devil wears prada 2"',
    '"devil wears prada 2"',
    "#devilwearsprada2",
    '"the devil wears prada"',
    '"miranda priestly"',
    '"anna wintour" prada',
    '"meryl streep" prada',
    '"anne hathaway" prada',
    '"emily blunt" prada',
    '"stanley tucci" prada',
    '"lady gaga" prada',
    '"doechii" prada',
]

SINCE_ISO = "2026-03-01T00:00:00.000Z"
UNTIL_ISO = "2026-05-21T23:59:59.000Z"


## 5. Run the collection

Each query is capped at 100 posts with a polite sleep of 0.6 seconds between pages. A short delay between queries spreads requests evenly across the rate-limit window.


In [7]:
all_rows = []

for q in queries:
    print(f"→ {q}")
    rows = collect_search_posts(
        client,
        q=q,
        limit_total=100,
        page_size=25,
        since_iso=SINCE_ISO,
        until_iso=UNTIL_ISO,
    )
    print(f"  collected: {len(rows)}")
    all_rows.extend(rows)
    time.sleep(1.0)

df_posts = pd.DataFrame(all_rows)
df_posts["created_at"] = pd.to_datetime(df_posts["created_at"], utc=True, errors="coerce")
df_posts = df_posts.sort_values("created_at", ascending=False).reset_index(drop=True)

print(f"\nRaw rows collected: {len(df_posts)}")


→ "the devil wears prada 2"
  collected: 100
→ "devil wears prada 2"
  collected: 100
→ #devilwearsprada2
  collected: 100
→ "the devil wears prada"
  collected: 100
→ "miranda priestly"
  collected: 100
→ "anna wintour" prada
  collected: 100
→ "meryl streep" prada
  collected: 100
→ "anne hathaway" prada
  collected: 100
→ "emily blunt" prada
  collected: 100
→ "stanley tucci" prada
  collected: 100
→ "lady gaga" prada
  collected: 100
→ "doechii" prada
  collected: 100

Raw rows collected: 1200


## 6. Deduplication

The twelve queries overlap by design (a post about Meryl Streep on the red carpet matches both `"meryl streep" prada` and `"the devil wears prada 2"`). Posts are deduplicated on `uri`. The overlap rate is itself a useful signal: a high duplicate rate confirms that the queries are converging on the same conversation.


In [8]:
df = df_posts.drop_duplicates(subset="uri").reset_index(drop=True)

print(f"Raw rows:        {len(df_posts)}")
print(f"Unique posts:    {len(df)}")
print(f"Unique authors:  {df['author_handle'].nunique()}")
print(f"Overlap rate:    {1 - len(df) / len(df_posts):.1%}")


Raw rows:        1200
Unique posts:    937
Unique authors:  729
Overlap rate:    21.9%


## 7. Per-query yield

Summary statistics per query: number of unique posts, unique authors, reply and like volumes. This table shows which queries are the strongest contributors and which audiences drive the most engagement.


In [9]:
query_summary = (
    df.groupby("query")
      .agg(
          n_posts=("uri", "count"),
          unique_authors=("author_handle", "nunique"),
          total_replies=("reply_count", "sum"),
          avg_replies=("reply_count", "mean"),
          total_likes=("like_count", "sum"),
          avg_likes=("like_count", "mean"),
      )
      .sort_values("n_posts", ascending=False)
      .round(2)
)
query_summary


,n_posts,unique_authors,total_replies,avg_replies,total_likes,avg_likes
query,,,,,,
"""anna wintour"" prada",96,87,47,0.49,398,4.15
"""doechii"" prada",95,82,18,0.19,204,2.15
"""miranda priestly""",95,91,41,0.43,524,5.52
#devilwearsprada2,95,86,23,0.24,234,2.46
"""lady gaga"" prada",89,72,28,0.31,439,4.93
"""stanley tucci"" prada",82,77,58,0.71,1112,13.56
"""anne hathaway"" prada",76,75,106,1.39,1440,18.95
"""emily blunt"" prada",74,65,35,0.47,390,5.27
"""meryl streep"" prada",66,59,23,0.35,153,2.32


## 8. Top posts by reply count

The most-replied posts in the pilot are the natural seeds for the reply graph constructed in the next stage of the project.


In [10]:
top_posts = df.sort_values("reply_count", ascending=False).head(20)
top_posts[[
    "query", "author_handle", "text", "created_at",
    "reply_count", "repost_count", "like_count", "uri",
]]


,query,author_handle,text,created_at,reply_count,repost_count,like_count,uri
383,"""anne hathaway"" prada",squeakyllama.bsky.social,I think a lot how in the first Devil Wears Pra...,2026-05-12 18:54:56.258000+00:00,67,59,915,at://did:plc:oxxpplof2s4fb263v5pcmrie/app.bsky...
265,"""emily blunt"" prada",macydaisy.bsky.social,My extroverted side made plans my introverted ...,2026-05-16 18:02:07.083000+00:00,15,2,121,at://did:plc:7ectns3getcrj2ff5ka2o5yk/app.bsky...
195,"""the devil wears prada 2""",mattielubchansky.com,screamed so loud (derogatory) at kara swisher ...,2026-05-19 01:42:15.207000+00:00,12,5,255,at://did:plc:pgppxekvpklq6smlilrxg7ke/app.bsky...
495,"""anna wintour"" prada",rosserford.bsky.social,Anna Wintour would almost certainly disapprove...,2026-05-10 13:51:07.249000+00:00,11,1,100,at://did:plc:dzf7v2jyvbw2lqb2dcce4xfa/app.bsky...
211,"""miranda priestly""",sachacoward.bsky.social,Extremely banal gym selfie insert Miranda Prie...,2026-05-18 08:02:15.878000+00:00,11,1,144,at://did:plc:hnt5kntls5mwva5dgdeyr52i/app.bsky...
667,"""stanley tucci"" prada",vandroidhelsing.bsky.social,sorry but nobody who was in The Devil Wears Pr...,2026-05-04 23:35:58.195000+00:00,9,22,541,at://did:plc:aht75weh3zbaihskxuv4wkgv/app.bsky...
646,#devilwearsprada2,originalsp.in,In honor of the period once known as APA Herit...,2026-05-05 16:08:39.919000+00:00,6,29,82,at://did:plc:ja5soujdv5a735xhr3cmohdv/app.bsky...
285,"""anne hathaway"" prada",funkelly.bsky.social,post a meme made by you \n\n“Are those…?”\n“Ta...,2026-05-16 02:45:37.536000+00:00,6,4,68,at://did:plc:ha4gmmdxg6h2prjjltwwcmds/app.bsky...
227,"""anne hathaway"" prada",zoewithasword.bsky.social,THIS WAS CONSIDERED BIGGER SIZED \n\nSHE WAS A...,2026-05-17 20:15:40.892000+00:00,6,5,99,at://did:plc:qyz6pl4auoos7cahbc42yaod/app.bsky...
636,"""lady gaga"" prada",popcrave.com,Meryl Streep says that if ‘The Devil Wears Pra...,2026-05-05 23:21:11.938000+00:00,5,16,175,at://did:plc:pcl72lxq7w2nrr7jkiks4qkc/app.bsky...


## 9. Hashtags and mentions

Hashtags and mentions extracted from the post facets provide a first view of the dominant themes and of the accounts most frequently referenced in the conversation. The mention edges will complement reply edges in the network graph.


In [11]:
top_hashtags = (
    df.explode("hashtags")
      .dropna(subset=["hashtags"])["hashtags"]
      .str.lower()
      .value_counts()
      .head(20)
)
print("Top 20 hashtags:")
print(top_hashtags)


Top 20 hashtags:
hashtags
devilwearsprada2       105
ladygaga                23
thedevilwearsprada2     22
doechii                 17
michael                 16
merylstreep             14
runway                  14
boxoffice               13
annehathaway            12
devilwearsprada          9
filmsky                  8
movies                   8
news                     7
star                     6
wars                     6
pedro                    6
pascal                   6
the                      6
devil                    6
wears                    6
Name: count, dtype: int64


In [12]:
mentions_exploded = df.explode("mentions").dropna(subset=["mentions"])
posts_with_mention = (df["mentions"].str.len() > 0).sum()

print(f"Posts with at least one mention: {posts_with_mention}")
print(f"Unique mentioned DIDs:           {mentions_exploded['mentions'].nunique()}")
print()
print("Top 10 mentioned DIDs:")
print(mentions_exploded["mentions"].value_counts().head(10))


Posts with at least one mention: 35
Unique mentioned DIDs:           32

Top 10 mentioned DIDs:
mentions
did:plc:spufbpsjkufpcwamze6htgue    7
did:plc:6q7yd6hstzajrywunssxma26    3
did:plc:upfuphuakf3wlmhnoidx7ho3    3
did:plc:irunns6swg7cwdonsdzhuslw    2
did:plc:v7dkbv75sa5qmmriu2mplri4    2
did:plc:3u7in7dx4a42qsv2n4ikzb6l    2
did:plc:dwaduijo6ngdgzmir5vovzrh    2
did:plc:i6k6scfcdaup4e2va33nkprb    1
did:plc:mvte3ogwuele6tndqgzy742q    1
did:plc:2rzitp7efk47tbbjbxzfoysi    1
Name: count, dtype: int64


## 10. Language distribution

The sentiment lexicons (AFINN, VADER, NRCLex) and the NER pipeline used downstream are English-only. The share of English posts determines how much of the pilot is directly usable and how much requires either language detection or translation.


In [13]:
def _primary_lang(langs):
    if isinstance(langs, (list, tuple)) and langs:
        return langs[0]
    return None

df["lang_primary"] = df["langs"].apply(_primary_lang)

lang_counts = df["lang_primary"].fillna("(none)").value_counts()
share_en = df["lang_primary"].fillna("").str.startswith("en").mean()

print("Top languages:")
print(lang_counts.head(10))
print(f"\nShare of posts tagged as English: {share_en:.1%}")


Top languages:
lang_primary
en        509
(none)    207
pt         69
en-US      41
es         39
fr         12
de          8
it-IT       8
pt-PT       8
id          6
Name: count, dtype: int64

Share of posts tagged as English: 58.7%


## 11. Reply structure

A post is part of a reply thread when `reply_parent_uri` is set. The split between root posts and reply posts, together with the declared `reply_count` aggregates, indicates how much actual discussion the topic generates and is the input to the conversation-graph reconstruction in the next notebook.


In [14]:
n_root = df["reply_parent_uri"].isna().sum()
n_reply = df["reply_parent_uri"].notna().sum()
declared_replies = int(df["reply_count"].fillna(0).sum())
posts_with_replies = int((df["reply_count"].fillna(0) > 0).sum())

print(f"Root posts:               {n_root}")
print(f"Reply posts:              {n_reply}")
print(f"Declared replies (sum):   {declared_replies}")
print(f"Posts with >= 1 reply:    {posts_with_replies}")


Root posts:               832
Reply posts:              105
Declared replies (sum):   451
Posts with >= 1 reply:    220


## 12. Save

The pilot is saved in two formats: a CSV with list fields serialised as pipe-separated strings (readable in any spreadsheet) and a parquet file that preserves the native list types for direct use in the following notebooks.


In [15]:
out_dir = Path("../data/raw")
out_dir.mkdir(parents=True, exist_ok=True)

df_csv = df.copy()
for col in ("hashtags", "mentions", "links", "langs", "labels"):
    df_csv[col] = df_csv[col].apply(
        lambda x: "|".join(map(str, x)) if isinstance(x, (list, tuple)) else ""
    )

df_csv.to_csv(out_dir / "bluesky_posts_pilot.csv", index=False)
query_summary.to_csv(out_dir / "bluesky_query_summary_pilot.csv")

# JSON Lines: preserves native list types, one post per line.
df.to_json(
    out_dir / "bluesky_posts_pilot.jsonl",
    orient="records",
    lines=True,
    date_format="iso",
    force_ascii=False,
)

print("Saved:")
for name in ("bluesky_posts_pilot.csv", "bluesky_query_summary_pilot.csv", "bluesky_posts_pilot.jsonl"):
    print(" -", out_dir / name)


Saved:
 - ../data/raw/bluesky_posts_pilot.csv
 - ../data/raw/bluesky_query_summary_pilot.csv
 - ../data/raw/bluesky_posts_pilot.jsonl


## 13. Pilot outcome

The pilot collected **937 unique posts** from **729 unique authors** over the three-month window, with **451 declared replies** distributed across **220 posts that received at least one reply**. The cross-query overlap of about 22 % confirms that the twelve queries converge on a single coherent conversation rather than on disjoint topics.

English is the primary language for **59 %** of the posts; the remaining share is dominated by Portuguese, Spanish, and French, all of which can be either filtered out or processed separately at the analysis stage.

Engagement is concentrated on a small number of high-visibility posts: the most replied-to post received 67 replies and 915 likes, and the `"anne hathaway" prada` and `"stanley tucci" prada` queries dominate by average likes per post (18.9 and 13.6 respectively), reflecting the press-tour spike captured by the time window. Hashtag activity is led by `#devilwearsprada2` (105 occurrences), `#ladygaga`, `#thedevilwearsprada2`, and `#doechii`, confirming both the film-centric and the press-tour-centric facets of the conversation.

Of the collected posts, 105 are themselves replies (with `reply_parent_uri` set), and 220 of the root posts have at least one declared reply. These 220 root posts are the seeds from which the full reply tree will be reconstructed in the next stage, where each thread is expanded and conversation edges between authors are extracted to build the network graph.
